In [2]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
    --------------------------------------- 1.6/101.7 MB 11.3 MB/s eta 0:00:09
   - -------------------------------------- 3.9/101.7 MB 11.7 MB/s eta 0:00:09
   -- ------------------------------------- 6.3/101.7 MB 11.7 MB/s eta 0:00:09
   --- ------------------------------------ 8.7/101.7 MB 11.7 MB/s eta 0:00:08
   ---- ----------------------------------- 11.0/101.7 MB 11.8 MB/s eta 0:00:08
   ----- ---------------------------------- 13.4/101.7 MB 11.8 MB/s eta 0:00:08
   ------ --------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.1.1 -> 26.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

sys.path.append('../src')
from preprocessing import load_and_preprocess, create_features, get_timeseries_split, prepare_lstm_windows
from metrics import calculate_metrics # Ensure metrics.py from previous step is in src/

In [ ]:
splits = [
    ("Split 1: 2000-2018", "2000-01-01", "2015-12-31", "2016-01-01", "2018-12-31"),
    ("Split 2: COVID Era", "2000-01-01", "2018-12-31", "2019-01-01", "2020-12-31"),
    ("Split 3: Recovery",  "2000-01-01", "2020-12-31", "2021-01-01", "2022-12-31"),
    ("Split 4: Modern",    "2000-01-01", "2022-12-31", "2023-01-01", "2024-12-31")
]

raw_df = load_and_preprocess('../data/data.csv')
ml_df = create_features(raw_df)

c:\Users\shris\OneDrive\Desktop\dl\notebooks\../src\preprocessing.py:15: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [4]:
results = []

for name, s_train, e_train, s_test, e_test in splits:
    print(f"Processing {name}...")
    
    # Prepare Data
    train, test = get_timeseries_split(ml_df, s_train, e_train, s_test, e_test)
    X_train, y_train = train[['Lag_1', 'Lag_2', 'MA_5']], train['Target']
    X_test, y_test = test[['Lag_1', 'Lag_2', 'MA_5']], test['Target']
    
    # 1. Linear Regression
    lr_model = LinearRegression().fit(X_train, y_train)
    lr_preds = lr_model.predict(X_test)
    results.append({"Model": "Linear Regression", "Split": name, **calculate_metrics(y_test, lr_preds)})
    
    # 2. XGBoost
    xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.05).fit(X_train, y_train)
    xgb_preds = xgb_model.predict(X_test)
    results.append({"Model": "XGBoost", "Split": name, **calculate_metrics(y_test, xgb_preds)})
    
    # 3. ARIMA (Simplified Walk-Forward)
    history = list(y_train[-200:]) # Use last 200 days for speed
    arima_preds = []
    for val in y_test[:50]: # Testing first 50 days for demonstration
        m = ARIMA(history, order=(5,1,0)).fit()
        arima_preds.append(m.forecast()[0])
        history.append(val)
    results.append({"Model": "ARIMA", "Split": name, **calculate_metrics(y_test[:50], arima_preds)})

    # 4. LSTM
    X_l_train, y_l_train, scaler = prepare_lstm_windows(train['Target'])
    X_l_test, y_l_test, _ = prepare_lstm_windows(test['Target'])
    
    lstm_model = Sequential([
        LSTM(50, return_sequences=True, input_shape=(X_l_train.shape[1], 1)),
        Dropout(0.2),
        LSTM(50),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')
    lstm_model.fit(X_l_train, y_l_train, epochs=5, batch_size=32, verbose=0)
    
    lstm_preds = scaler.inverse_transform(lstm_model.predict(X_l_test))
    results.append({"Model": "LSTM", "Split": name, **calculate_metrics(y_test[60:], lstm_preds.flatten())})

# Convert to DataFrame for easy comparison[cite: 2]
results_df = pd.DataFrame(results)
print(results_df)

Processing Split 1: 2000-2018...


C:\Users\shris\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step
Processing Split 2: COVID Era...


C:\Users\shris\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step
Processing Split 3: Recovery...


C:\Users\shris\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Processing Split 4: Modern...


C:\Users\shris\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step
                Model               Split           MAE          RMSE  \
0   Linear Regression  Split 1: 2000-2018     51.550560     67.522439   
1             XGBoost  Split 1: 2000-2018    936.715145   1264.273627   
2               ARIMA  Split 1: 2000-2018     75.029256     93.561370   
3                LSTM  Split 1: 2000-2018   4161.432664   4245.412926   
4   Linear Regression  Split 2: COVID Era     93.682692    144.068157   
5             XGBoost  Split 2: COVID Era    352.199236    581.360233   
6               ARIMA  Split 2: COVID Era     57.572132     74.111649   
7                LSTM  Split 2: COVID Era   4372.939322   4445.190631   
8   Linear Regression   Split 3: Recovery    113.072050    149.718647   
9             XGBoost   Split 3: Recovery   3115.315291   3353.532712   
10              ARIMA   Split 3: Recovery    151.621518    202.719815   
11               LSTM   Split 3: Recovery   7853.074711   8041.229079   
12  Linear 

In [ ]:
# 1. Calculate Average Performance across all splits
summary_metrics = results_df.groupby('Model')[['MAE', 'RMSE', 'MAPE', 'R2', 'DA']].mean()

# 2. Identify the best model based on Lowest Mean Absolute Percentage Error (MAPE)
# MAPE is often preferred for stock prices as it provides a percentage-based error
best_by_accuracy = summary_metrics['MAPE'].idxmin()

# 3. Identify the best model based on Highest Directional Accuracy (DA)
# DA is critical for stock prediction as it measures the ability to predict the correct direction[cite: 2]
best_by_direction = summary_metrics['DA'].idxmax()

# 4. Calculate Robustness (Stability)[cite: 2]
# A robust model will have the lowest variance (standard deviation) in error across market splits[cite: 2]
robustness_scores = results_df.groupby('Model')['RMSE'].std()
most_robust_model = robustness_scores.idxmin()

print("--- Final Model Comparison (Averages) ---")
print(summary_metrics)
print("\n--- Decision Logic ---")
print(f"Best Model (Lowest Price Error): {best_by_accuracy}")
print(f"Best Model (Highest Directional Accuracy): {best_by_direction}")
print(f"Most Robust Model (Highest Consistency across splits): {most_robust_model}")

# Determine the Overall Winner
# To rank in the top five, prioritize the model that is both accurate and consistent[cite: 2]
overall_winner = most_robust_model if most_robust_model == best_by_accuracy else best_by_accuracy
print(f"\nRecommended Best Overall Model: {overall_winner}")

--- Final Model Comparison (Averages) ---
                           MAE         RMSE       MAPE         R2         DA
Model                                                                       
ARIMA                98.072026   126.347115   0.799929   0.765200  59.693878
LSTM               7077.115293  7221.905647  47.004595 -27.613206  54.215913
Linear Regression    91.992225   129.051828   0.657455   0.990059  59.038270
XGBoost            1837.835907  2248.391240  10.722416  -1.717620  11.570934

--- Decision Logic ---
Best Model (Lowest Price Error): Linear Regression
Best Model (Highest Directional Accuracy): ARIMA
Most Robust Model (Highest Consistency across splits): Linear Regression

Recommended Best Overall Model for Submission: Linear Regression
